# QA 09 — regresión polinómica y deformación de curvas

Curvas 1D, potencias transformadas, interpolación sintética, replay incremental y límites de grado alto.

> Run this notebook from top to bottom after installing the project with
> `pip install -e ".[notebooks]"`. Figures are genuine public-API outputs.
> Cells intentionally contain no assertions: automated invariants live in
> `tests/`, while this notebook is for human visual inspection.

In [ ]:
from pathlib import Path
import sys

candidate = Path.cwd().resolve()
while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
if not (candidate / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from inside the Mlektic repository.")
ROOT = candidate
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from IPython.display import display
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from mlektic import visualize_lr
from notebooks._support import case_heading

SEED = 42
rng = np.random.default_rng(SEED)

## Qué valida este notebook

Una regresión polinómica conserva una entrada original 1D, pero transforma esa
entrada en términos como `x`, `x²`, `x³`, etc. Por eso la geometría visible debe
ser una **curva**, no una recta, y la ecuación superior debe mostrar las
potencias que realmente consume el estimador.

Revise en cada caso:

1. que el eje X use el `feature_names` solicitado;
2. que la línea del modelo tenga curvatura visible;
3. que la fórmula simbólica y la sustitución animada usen `x₁`, `x₁²`, `x₁³`, etc.;
4. que el último estado coincida con el modelo ajustado;
5. que grados altos se abrevien con `…` sin desbordar el lienzo.

Con `Ridge`/`LinearRegression`, los estados son una interpolación sintética
desde la línea base hasta la curva final. Con `SGDRegressor`, Mlektic reconstruye
un replay incremental y reserva el último estado para el estimador suministrado.

In [ ]:
X_quadratic = np.linspace(-2.8, 2.8, 110).reshape(-1, 1)
y_quadratic = 3.0 - 1.2 * X_quadratic[:, 0] + 2.4 * X_quadratic[:, 0] ** 2

X_waves = np.linspace(-3.0, 3.0, 180).reshape(-1, 1)
y_waves = (
    35 * np.sin(2.7 * X_waves[:, 0])
    + 16 * np.cos(5.2 * X_waves[:, 0])
    + 5 * X_waves[:, 0] ** 2
    + rng.normal(0, 4, len(X_waves))
)

diabetes = load_diabetes()
X_bmi = diabetes.data[:, [2]]
y_diabetes = diabetes.target

print('Parábola:', X_quadratic.shape)
print('Ondas:', X_waves.shape)
print('Diabetes/BMI:', X_bmi.shape)

### `POLY-QUADRATIC-POWERS`

**Inspect:** parábola inequívoca, eje descriptivo posición y ecuaciones canónicas x₁ y x₁²

In [ ]:
case_heading("POLY-QUADRATIC-POWERS", "parábola inequívoca, eje descriptivo posición y ecuaciones canónicas x₁ y x₁²")
quadratic = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression(),
).fit(X_quadratic, y_quadratic)
display(visualize_lr(
    quadratic, X_quadratic, y_quadratic,
    steps=16, max_frames=8, animation_mode='hybrid', interpolation_frames=4,
    detail='complete', feature_names=['posición'],
    title='QA polinómica: parábola y potencias explícitas', size='wide',
))

### `POLY-CUBIC-SCALED-DIABETES`

**Inspect:** curva cúbica real con eje BMI y ecuaciones canónicas x₁, x₁² y x₁³

In [ ]:
case_heading("POLY-CUBIC-SCALED-DIABETES", "curva cúbica real con eje BMI y ecuaciones canónicas x₁, x₁² y x₁³")
cubic_bmi = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False),
    StandardScaler(),
    Ridge(alpha=4.0),
).fit(X_bmi, y_diabetes)
display(visualize_lr(
    cubic_bmi, X_bmi, y_diabetes,
    steps=20, max_frames=10, mode='final_interp',
    animation_mode='hybrid', interpolation_frames=4, fps=35,
    metrics=['mse', 'mae', 'r2'], detail='complete',
    feature_names=['BMI'], title='QA polinómica: eje BMI y potencias de x₁',
    theme='academic', size='wide',
))

### `POLY-WAVY-HIGH-DEGREE`

**Inspect:** deformación extrema desde una línea base hasta una curva con muchas oscilaciones

In [ ]:
case_heading("POLY-WAVY-HIGH-DEGREE", "deformación extrema desde una línea base hasta una curva con muchas oscilaciones")
wavy = make_pipeline(
    PolynomialFeatures(degree=14, include_bias=False),
    StandardScaler(),
    Ridge(alpha=0.08),
).fit(X_waves, y_waves)
display(visualize_lr(
    wavy, X_waves, y_waves,
    steps=40, max_frames=20, mode='final_interp',
    animation_mode='hybrid', interpolation_frames=5, fps=35,
    metrics=['mse', 'mae', 'r2'], detail='academic',
    feature_names=['posición'],
    title='QA polinómica: múltiples oscilaciones y grado 14 abreviado',
    theme='academic', size='wide',
))

### `POLY-ITERATIVE-REPLAY`

**Inspect:** aproximaciones reconstruidas de un polinomio incremental y cierre en el estimador exacto

In [ ]:
case_heading("POLY-ITERATIVE-REPLAY", "aproximaciones reconstruidas de un polinomio incremental y cierre en el estimador exacto")
iterative_poly = make_pipeline(
    PolynomialFeatures(degree=5, include_bias=False),
    StandardScaler(),
    SGDRegressor(max_iter=1200, tol=1e-4, random_state=SEED),
).fit(X_waves, y_waves)
display(visualize_lr(
    iterative_poly, X_waves, y_waves,
    steps=24, max_frames=12, mode='iterative', animation_mode='native',
    smooth='ema', smooth_beta=0.9,
    metrics=['mse', 'mae', 'r2'], detail='complete',
    feature_names=['posición'],
    title='QA polinómica: replay incremental reconstruido',
    theme='accessible', size='wide',
))

## Lectura esperada

- Los tres primeros casos no afirman mostrar pasos privados de entrenamiento:
  el subtítulo debe decir **Synthetic interpolation**.
- El caso incremental debe decir **Reconstructed replay** y terminar en
  **fitted**.
- `smooth='ema'` sólo afecta la pérdida del replay incremental. No se aplica a
  la interpolación sintética, que conserva el extremo exacto.
- En grado 14, la ecuación superior debe mostrar las primeras potencias,
  `\cdots` y la potencia 14; la curva completa sí usa todos los coeficientes.

## Canvas-size coverage

Run the same representative figure in every supported size. Inspect clipping,
mathematical spacing, controls, legends, axes, and any below-plot derivation.

### `POLY-SIZE-DEFAULT`

**Inspect:** default canvas preset for this notebook's representative figure

In [ ]:
case_heading("POLY-SIZE-DEFAULT", "default canvas preset for this notebook's representative figure")
X_size=np.linspace(-2.8,2.8,110).reshape(-1,1)
y_size=3.0-1.2*X_size[:,0]+2.4*X_size[:,0]**2
m_size=make_pipeline(PolynomialFeatures(degree=2,include_bias=False),LinearRegression()).fit(X_size,y_size)
display(visualize_lr(m_size,X_size,y_size,steps=7,max_frames=5,feature_names=['posición'],size='default'))

### `POLY-SIZE-COMPACT`

**Inspect:** compact canvas preset for this notebook's representative figure

In [ ]:
case_heading("POLY-SIZE-COMPACT", "compact canvas preset for this notebook's representative figure")
X_size=np.linspace(-2.8,2.8,110).reshape(-1,1)
y_size=3.0-1.2*X_size[:,0]+2.4*X_size[:,0]**2
m_size=make_pipeline(PolynomialFeatures(degree=2,include_bias=False),LinearRegression()).fit(X_size,y_size)
display(visualize_lr(m_size,X_size,y_size,steps=7,max_frames=5,feature_names=['posición'],size='compact'))

### `POLY-SIZE-NOTEBOOK`

**Inspect:** notebook canvas preset for this notebook's representative figure

In [ ]:
case_heading("POLY-SIZE-NOTEBOOK", "notebook canvas preset for this notebook's representative figure")
X_size=np.linspace(-2.8,2.8,110).reshape(-1,1)
y_size=3.0-1.2*X_size[:,0]+2.4*X_size[:,0]**2
m_size=make_pipeline(PolynomialFeatures(degree=2,include_bias=False),LinearRegression()).fit(X_size,y_size)
display(visualize_lr(m_size,X_size,y_size,steps=7,max_frames=5,feature_names=['posición'],size='notebook'))

### `POLY-SIZE-WIDE`

**Inspect:** wide canvas preset for this notebook's representative figure

In [ ]:
case_heading("POLY-SIZE-WIDE", "wide canvas preset for this notebook's representative figure")
X_size=np.linspace(-2.8,2.8,110).reshape(-1,1)
y_size=3.0-1.2*X_size[:,0]+2.4*X_size[:,0]**2
m_size=make_pipeline(PolynomialFeatures(degree=2,include_bias=False),LinearRegression()).fit(X_size,y_size)
display(visualize_lr(m_size,X_size,y_size,steps=7,max_frames=5,feature_names=['posición'],size='wide'))

### `POLY-SIZE-CLASSROOM`

**Inspect:** classroom canvas preset for this notebook's representative figure

In [ ]:
case_heading("POLY-SIZE-CLASSROOM", "classroom canvas preset for this notebook's representative figure")
X_size=np.linspace(-2.8,2.8,110).reshape(-1,1)
y_size=3.0-1.2*X_size[:,0]+2.4*X_size[:,0]**2
m_size=make_pipeline(PolynomialFeatures(degree=2,include_bias=False),LinearRegression()).fit(X_size,y_size)
display(visualize_lr(m_size,X_size,y_size,steps=7,max_frames=5,feature_names=['posición'],size='classroom'))